# Shrub Lists Extras — Standardization

Standardize original and revised shrub-list CSVs into a **canonical schema** for downstream use.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import re


In [ ]:
ORIGINAL_CSV = Path('output_original') / 'shrub_list.csv'   # edit if needed
REVISED_CSV  = Path('output_revised')  / 'shrub_list.csv'   # edit if needed
SITE_ID = 'DL_Bliss'
SCAN_ID = 'CAAEU_0001_20250722_1'


In [ ]:
CANONICAL_TARGETS = [
    "shrub_id", "x", "y", "z", "height_m", "radius_m", "diameter_m", "area_m2", "volume_m3"
]

ALIASES = {
    "shrub_id":  ["shrub_id", "id", "object_id", "patch_id"],
    "x":         ["x", "x_coord", "xcenter", "centroid_x", "utm_x", "easting", "lon", "longitude"],
    "y":         ["y", "y_coord", "ycenter", "centroid_y", "utm_y", "northing", "lat", "latitude"],
    "z":         ["z", "z_coord", "centroid_z", "elevation", "height_base"],
    "height_m":  ["height_m", "height", "shrub_height", "max_height", "h", "height.m"],
    "radius_m":  ["radius_m", "radius", "r", "crown_radius"],
    "diameter_m":["diameter_m", "diameter", "width", "crown_diameter"],
    "area_m2":   ["area_m2", "area", "footprint_area", "shrub_area"],
    "volume_m3": ["volume_m3", "volume", "shrub_volume"],
}

def normalize_name(s: str) -> str:
    s = s.strip().lower()
    s = re.sub(r"[^a-z0-9]+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return s

def inspect_csv(path: Path, n=5):
    df = pd.read_csv(path)
    print("path:", path)
    print("shape:", df.shape)
    print("columns:", list(df.columns))
    display(df.head(n))
    return df

def build_column_map(columns):
    norm_to_orig = {normalize_name(c): c for c in columns}
    col_map = {}
    for canon, aliases in ALIASES.items():
        found = None
        for a in aliases:
            a_norm = normalize_name(a)
            if a_norm in norm_to_orig:
                found = norm_to_orig[a_norm]
                break
        col_map[canon] = found
    return col_map


In [ ]:
def standardize_shrub_list(path, *, workflow, site_id, scan_id, verbose=True):
    path = Path(path)
    df = pd.read_csv(path)
    col_map = build_column_map(df.columns)

    out = pd.DataFrame()
    for canon in CANONICAL_TARGETS:
        src = col_map.get(canon)
        out[canon] = df[src] if src is not None else np.nan

    if out["diameter_m"].isna().all() and not out["radius_m"].isna().all():
        out["diameter_m"] = 2.0 * out["radius_m"]
    if out["radius_m"].isna().all() and not out["diameter_m"].isna().all():
        out["radius_m"] = 0.5 * out["diameter_m"]

    for c in ["x","y","z","height_m","radius_m","diameter_m","area_m2","volume_m3"]:
        out[c] = pd.to_numeric(out[c], errors="coerce")

    if out["shrub_id"].isna().all():
        out["shrub_id"] = [f"{workflow}_{i:06d}" for i in range(len(out))]

    out["source_workflow"] = workflow
    out["site_id"] = site_id
    out["scan_id"] = scan_id
    out["source_path"] = str(path)

    if verbose:
        print("column map:", col_map)
        display(out.head())

    return out

def save_standardized(df, out_path):
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(out_path, index=False)
    print("saved:", out_path)


In [ ]:
# Example usage:
# df_orig_raw = inspect_csv(ORIGINAL_CSV)
# df_rev_raw = inspect_csv(REVISED_CSV)
#
# shrubs_orig = standardize_shrub_list(ORIGINAL_CSV, workflow='original', site_id=SITE_ID, scan_id=SCAN_ID)
# shrubs_rev  = standardize_shrub_list(REVISED_CSV,  workflow='revised',  site_id=SITE_ID, scan_id=SCAN_ID)
#
# save_standardized(shrubs_orig, 'standardized/shrubs_original_standardized.csv')
# save_standardized(shrubs_rev,  'standardized/shrubs_revised_standardized.csv')
